In [1]:
# Modules
import numpy as np
import pandas as pd
import itertools
import destruction_models as models
import tensorflow as tf
import random

from destruction_utilities import *
from destruction_statistics import *
from numpy import random
import matplotlib.pyplot as plt
from tensorflow.keras import callbacks, preprocessing
from tensorflow.keras.utils import Sequence
from tensorflow.keras.metrics import CategoricalAccuracy, Precision, AUC
from tensorflow.keras.models import load_model
from sklearn.metrics import precision_recall_curve, roc_auc_score
from os import path
import zarr
import shutil
from tensorflow.keras import backend as K
import gc
import time
import pickle

In [2]:
CITY = 'aleppo'
TILE_SIZE = (128,128)
BATCH_SIZE = 32

In [3]:
def get_zarr(city, type, set, balanced = False):
    if balanced:
        path = f'../data/{city}/others/{city}_{type}s_{set}_balanced.zarr'
    else:
        path = f'../data/{city}/others/{city}_{type}s_{set}.zarr'
    print(path)
    return zarr.open(path, mode='r')


def get_path(city, type, set, balanced = False):
    if balanced:
        path = f'../data/{city}/others/{city}_{type}s_{set}_balanced.zarr'
    else:
        path = f'../data/{city}/others/{city}_{type}s_{set}.zarr'
    return path

def reshuffle_data(X, y): 
    print("Shuffling data started...")
    
    balanced_path_images = get_path(CITY, 'image', 'train', balanced=True)
    balanced_path_labels = get_path(CITY, 'label', 'train', balanced=True)
    shuffled_path_images = get_path(CITY, 'image', 'train', balanced=True).split('.zarr')[0] + "_shuffled.zarr"
    shuffled_path_labels = get_path(CITY, 'label', 'train', balanced=True).split('.zarr')[0] + "_shuffled.zarr"
    
    
    shutil.rmtree(shuffled_path_images, ignore_errors=True)
    shutil.rmtree(shuffled_path_labels, ignore_errors=True)
    n=X.shape[0]
    
    tuple_pair = make_tuple_pair(n, 1000)
    
    np.random.shuffle(tuple_pair)
    
    zarr.save(shuffled_path_images, np.empty((0,*TILE_SIZE,3)))
    zarr.save(shuffled_path_labels, np.empty((0,1,1,1)))
    
    X1 = zarr.open(shuffled_path_images, mode='a')
    y1 = zarr.open(shuffled_path_labels, mode='a')
    
    print(f"Reordering array in batches of 250. Total {len(tuple_pair)} sets..")
    for i, t in enumerate(tuple_pair):
        if i % 50 == 0 and i!=0:
            print(f"Finished {i} sets")
        Xs = X[t[0]:t[1]]
        ys = y[t[0]:t[1]]
        X1.append(Xs)
        y1.append(ys)
    
    
    shutil.rmtree(balanced_path_images)
    shutil.rmtree(balanced_path_labels)
    
    zarr.save(balanced_path_images, np.empty((0,*TILE_SIZE,3)))
    zarr.save(balanced_path_labels, np.empty((0,1,1,1)))
    
    X_shuffled = zarr.open(balanced_path_images, mode='a')
    y_shuffled = zarr.open(balanced_path_labels, mode='a')
    
    tuple_pair = make_tuple_pair(n, 7500)
    print(f"Shuffling array in batches of 10000. Total {len(tuple_pair)} sets..")
    for i, t in enumerate(tuple_pair):
        if i % 5 == 0 and i != 0:
            print(f"Finished {i} sets")
        shuffled = np.arange(0, 10000)
        np.random.shuffle(shuffled)
        Xs = X1[t[0]:t[1]][shuffled]
        ys = y1[t[0]:t[1]][shuffled]
        X_shuffled.append(Xs)
        y_shuffled.append(ys)
        
    shutil.rmtree(shuffled_path_images)
    shutil.rmtree(shuffled_path_labels)
    return X_shuffled, y_shuffled

def recall_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    recall = true_positives / (possible_positives + K.epsilon())
    return recall

def precision_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    return precision

def f1_m(y_true, y_pred):
    precision = precision_m(y_true, y_pred)
    recall = recall_m(y_true, y_pred)
    return 2*((precision*recall)/(precision+recall+K.epsilon()))

auc = AUC(
    num_thresholds=200,
    curve='ROC',
)
    
def run_model(training_images, training_labels, validation_images, validation_labels, epochs=50):
    training_generator = ZarrGenerator(training_images, training_labels, batch_size=BATCH_SIZE)
    validation_generator = ZarrGenerator(validation_images, validation_labels, batch_size=BATCH_SIZE)
    
    training_callbacks = [
#         callbacks.EarlyStopping(monitor='val_auc', restore_best_weights=True, patience=5),
    #     callbacks.BackupAndRestore(backup_dir='../models')
    ]

    args  = dict(shape=(*TILE_SIZE, 3), filters=16, units=32, dropout=0.1) # ! Check parameters before run
    model = models.convolutional_network(**args)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', f1_m, precision_m, recall_m, auc ])

    # Train model on dataset
    history = model.fit_generator(
        training_generator,
        validation_data=validation_generator, 
        epochs=epochs, 
        callbacks = training_callbacks
    )
    
    return model, history


def calculate_auc(model, test_images, test_labels):
    gc.collect(generation=2)    
    batch_size = 5000
    iters = test_images.shape[0] // batch_size
    preds = []
    labels = []
    for i in range(0, iters):
        end = (i+1)*batch_size
        if i == iters - 1:
            preds.append(model.predict(test_images[i*batch_size:]))
            labels.append(test_labels[i*batch_size:])
        else:
            preds.append(model.predict(test_images[i*batch_size: end]))
            labels.append(test_labels[i*batch_size: end])
            
    yhat = np.squeeze(np.concatenate(preds, axis=0))
    y = np.squeeze(np.concatenate(labels, axis=0 ))
    roc_auc = roc_auc_score(y, yhat)
    
    return roc_auc
    

Metal device set to: Apple M1


2022-06-27 21:13:08.289822: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2022-06-27 21:13:08.289924: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [4]:
class ZarrGenerator(Sequence):
    def __init__(self, images, labels, batch_size=BATCH_SIZE):
        self.images = images
        self.labels = labels
        self.batch_size = batch_size
        
    def __len__(self):
        return len(self.images)//self.batch_size
    
    def __getitem__(self, index):

        X = self.images[index*self.batch_size:(index+1)*self.batch_size]
        y = self.labels[index*self.batch_size:(index+1)*self.batch_size]
        
        return self.augment(X), y.flatten()
    
    def augment(self, X):
        # Horizontal and vertical flip
        flipping_funcs = [
            lambda image: image,
            lambda image: np.fliplr(image),
            lambda image: np.flipud(image),
            lambda image: np.flipud(np.fliplr(image))
        ]
        func = random.choice(flipping_funcs)
        X = func(X)
        
        # Brightness
        alpha = random.choice(np.linspace(0.85, 1.4))
        X = X * alpha
        
        return X

In [5]:
training_images = get_zarr(CITY, 'image', 'train', balanced=True)
training_labels = get_zarr(CITY, 'label', 'train', balanced=True)
validation_images = get_zarr(CITY, 'image', 'validate')
validation_labels = get_zarr(CITY, 'label', 'validate')
test_images = get_zarr(CITY, 'image', 'test')
test_labels = get_zarr(CITY, 'label', 'test')

../data/aleppo/others/aleppo_images_train_balanced.zarr
../data/aleppo/others/aleppo_labels_train_balanced.zarr
../data/aleppo/others/aleppo_images_validate.zarr
../data/aleppo/others/aleppo_labels_validate.zarr
../data/aleppo/others/aleppo_images_test.zarr
../data/aleppo/others/aleppo_labels_test.zarr


In [6]:
# print(training_images.shape, training_labels.shape, np.unique(training_labels, return_counts=True))
# print(validation_images.shape, validation_labels.shape, np.unique(validation_labels, return_counts=True))
# print(test_images.shape, test_labels.shape, np.unique(test_labels, return_counts=True))

In [ ]:
for i in range(0,50):
#     training_images_shuffled, training_labels_shuffled = reshuffle_data(training_images, training_labels)
    print("Shuffling complete..")
    model = run_model(training_images, training_labels, validation_images, validation_labels, epochs=50)
    history = model[1]
    model = model[0]
    print("Model optimization complete..")
    auc = calculate_auc(model, test_images, test_labels)
    
    with open(f'../models/run_{i}_hist', 'wb') as file_pi:
        pickle.dump(history.history, file_pi)
    
    if(auc > 0.8):
        ts = str(time.time())
        model.save(f'../models/{CITY}_{ts}', save_format="h5")

Shuffling complete..
Epoch 1/50


/var/folders/z1/7pydgng971nfzdf2rwg_p__r0000gn/T/ipykernel_90450/272411841.py:111: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  history = model.fit_generator(
2022-06-27 21:13:15.951330: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz
2022-06-27 21:13:16.813650: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


8037/8037 [==============================] - ETA: 0s - loss: 0.6652 - accuracy: 0.6120 - f1_m: 0.5953 - precision_m: 0.5942 - recall_m: 0.6658 - auc: 0.6390

2022-06-27 21:22:15.297048: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
